# GLM-OCR invoices — evaluation on the test set

Evaluates the best checkpoint (lowest evaluation loss) on the **801 held-out test invoices**.

**Protocol.** Greedy decoding, same prompt as training. Each field is normalised (lower-case, whitespace collapsed,
punctuation removed) and compared with the annotation. A wrong value counts as one false negative **and** one false
positive; an empty prediction counts as a false negative.

**Results (output of this notebook).**

| Field | Precision | Recall | F1 |
|---|---|---|---|
| CLIENT | 97.3 % | 97.3 % | 97.3 % |
| FOURNISSEUR | 98.3 % | 98.3 % | 98.3 % |
| DATE_FACTURE | 97.1 % | 97.1 % | 97.1 % |
| MONTANT_TTC | 97.0 % | 97.0 % | 97.0 % |
| NUMERO_FACTURE | 97.4 % | 97.4 % | 97.4 % |
| **Macro average** | **97.40 %** | **97.40 %** | **97.40 %** |

**Exact match** (all five fields correct on a document): **89.89 %**. Average inference time: **≈ 4.9 s / invoice** on a T4.

> **Limitation.** The test split was also used to select the best checkpoint during training (there was no separate
> validation split), so these scores may be slightly optimistic.

> **Environment.** Kaggle notebook, NVIDIA Tesla T4 (16 GB), Python 3.12. GLM-OCR requires the development
> version of `transformers` (installed in the first cell). After the first cell, **restart the kernel**.
>
> **Data.** The datasets are private (client documents) and are not distributed. Paths of the form
> `/kaggle/input/datasets/<KAGGLE_USER>/...` refer to private Kaggle datasets and must be adapted.

> **Anonymisation.** Outputs containing client or supplier data and document images were removed.
> Only aggregate metrics are kept.

## 1. Environment setup (restart the kernel afterwards)

In [ ]:
import os, sys, subprocess, shutil, site
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"]        = "1"

def pip(cmd):
    r = subprocess.run([sys.executable, "-m", "pip"] + cmd,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("ERR:", r.stderr[-300:])
    return r

pip(["uninstall", "-y", "transformers", "tokenizers",
     "huggingface-hub", "bitsandbytes", "peft", "trl", "accelerate"])

r = pip(["install", "-q", "--no-cache-dir",
         "git+https://github.com/huggingface/transformers.git"])
print(("OK" if r.returncode==0 else "ERR"), "transformers @ git")

for p in ["huggingface-hub==0.28.1", "accelerate==1.3.0",
          "peft==0.13.2", "trl==0.11.4"]:
    r = pip(["install", "-q", "--no-cache-dir", p])
    print(("OK" if r.returncode==0 else "ERR"), p)

pip(["uninstall", "-y", "bitsandbytes"])
print("✅ bitsandbytes supprimé")

for sp in site.getsitepackages():
    bnb_file = Path(sp) / "peft" / "tuners" / "lora" / "bnb.py"
    if bnb_file.exists():
        bnb_file.write_text(
            "# patched\n"
            "def dispatch_bnb_8bit(*a, **k): return None\n"
            "def dispatch_bnb_4bit(*a, **k): return None\n"
        )
        print("✅ peft/bnb.py patché")
        break

for sp in site.getsitepackages():
    tv_dir = Path(sp) / "torchvision"
    if not tv_dir.exists(): continue
    (tv_dir / "extension.py").write_text(
        "_HAS_OPS = False\n"
        "def _has_ops(): return False\n"
        "def _check_cuda_version(): pass\n"
        "def _load_library(n): return None\n"
        "def _assert_has_ops(): return None\n"
    )
    (tv_dir / "_meta_registrations.py").write_text("# patched\n")
    (tv_dir / "__init__.py").write_text(
        '__version__ = "0.0.0-patched"\n'
        "from .extension import _HAS_OPS\n"
        "try:\n    from . import transforms\nexcept: transforms = None\n"
        "try:\n    from . import io\nexcept: io = None\n"
    )
    print("✅ torchvision patché")
    break

print("\n⚠️  RESTART KERNEL maintenant !")

OK transformers @ git
OK huggingface-hub==0.28.1
OK accelerate==1.3.0
OK peft==0.13.2
OK trl==0.11.4
✅ bitsandbytes supprimé
✅ peft/bnb.py patché
✅ torchvision patché

⚠️  RESTART KERNEL maintenant !


## 2. Evaluation on the test set

In [ ]:
import json, re, warnings, logging, torch
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
from collections import defaultdict

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

# ── Config ────────────────────────────────────────────────
BEST_CKPT  = Path('/kaggle/input/datasets/<KAGGLE_USER>/glmocr-v7s7-best-ep13')
BASE_DATA  = Path('/kaggle/input/datasets/<KAGGLE_USER>/glmdata')
TEST_JSON  = BASE_DATA / 'output' / 'test_glmocr.json'
IMAGES_DIR = BASE_DATA / 'images'
MODEL_NAME = 'zai-org/GLM-OCR'
MAX_SIZE   = 1024
ENTITIES   = ['CLIENT', 'FOURNISSEUR', 'DATE_FACTURE', 'MONTANT_TTC', 'NUMERO_FACTURE']

PROMPT = (
    "Extrais les informations suivantes de cette facture en JSON :\n"
    "- CLIENT : nom du client\n"
    "- FOURNISSEUR : nom du fournisseur\n"
    "- DATE_FACTURE : date de la facture (format JJ/MM/AAAA)\n"
    "- MONTANT_TTC : montant total TTC (ou N/A si absent)\n"
    "- NUMERO_FACTURE : numero de facture\n"
    "Reponds UNIQUEMENT avec un objet JSON valide.\n"
    "Si une information est absente, mettre N/A."
)

gpu_ok = torch.cuda.is_available()
print(f'GPU : {torch.cuda.get_device_name(0) if gpu_ok else "CPU"}')

# ── Indexer images ────────────────────────────────────────
IMAGE_INDEX = {}
for p in IMAGES_DIR.rglob('*'):
    if p.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        IMAGE_INDEX[p.name] = p
print(f'Images indexées : {len(IMAGE_INDEX)}')

# ── Charger modèle ────────────────────────────────────────
print(f'\n⏳ Chargement modèle depuis {BEST_CKPT.name}...')

import peft
bnb_file = Path(peft.__file__).parent / 'tuners' / 'lora' / 'bnb.py'
if bnb_file.exists():
    bnb_file.write_text(
        '# disabled\n'
        'def dispatch_bnb_8bit(*a, **k): return None\n'
        'def dispatch_bnb_4bit(*a, **k): return None\n'
    )
import peft.tuners.lora.model as lora_model
lora_model.is_bnb_available      = lambda: False
lora_model.is_bnb_4bit_available = lambda: False

processor = AutoProcessor.from_pretrained(
    MODEL_NAME, use_fast=False, trust_remote_code=True)

base = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if gpu_ok else torch.float32,
    device_map='auto' if gpu_ok else None,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, str(BEST_CKPT), is_trainable=False)
model.eval()
print('✅ Modèle chargé')

# ── Charger test set ──────────────────────────────────────
with open(TEST_JSON) as f:
    test_data = json.load(f)

# Fixer chemins
test_fixed = []
for item in test_data:
    img = item['images'][0]
    if Path(img).exists():
        test_fixed.append(item)
        continue
    img_name = img.replace('\\', '/').split('/')[-1]
    if img_name in IMAGE_INDEX:
        it = dict(item)
        it['images'] = [str(IMAGE_INDEX[img_name])]
        test_fixed.append(it)

print(f'Test set : {len(test_fixed)} exemples')

# ── Fonctions normalisation ───────────────────────────────
def normalize(text):
    if not text or str(text).strip() in ['', 'N/A', 'null', 'None']:
        return ''
    t = str(text).strip().lower()
    t = re.sub(r'\s+', ' ', t)
    t = re.sub(r'[.,;:!?]', '', t)
    return t.strip()

def extract_json(text):
    try:
        text = text.strip()
        # Chercher JSON dans le texte
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except:
        pass
    try:
        return json.loads(text)
    except:
        return {}

def inference(img_path):
    try:
        image = Image.open(img_path).convert('RGB')
        w, h  = image.size
        if max(w, h) > MAX_SIZE:
            ratio = MAX_SIZE / max(w, h)
            image = image.resize((int(w*ratio), int(h*ratio)), Image.LANCZOS)

        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text":  PROMPT},
        ]}]

        inputs = processor.apply_chat_template(
            messages, tokenize=True,
            add_generation_prompt=True,
            return_dict=True, return_tensors='pt',
            processor_kwargs={"return_mm_token_type_ids": True},
        )
        inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v
                  for k, v in inputs.items()}
        inputs.pop('token_type_ids', None)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=None,
                top_p=None,
            )

        input_len = inputs['input_ids'].shape[1]
        generated = output[0][input_len:]
        text = processor.decode(generated, skip_special_tokens=True)
        return extract_json(text)

    except Exception as e:
        print(f'   ⚠️  Erreur inférence : {e}')
        return {}

# ── Évaluation ────────────────────────────────────────────
print(f'\n🔍 Évaluation sur {len(test_fixed)} exemples...\n')

tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)

errors        = 0
results_detail = []

for idx, item in enumerate(test_fixed):
    if idx % 50 == 0:
        print(f'   [{idx}/{len(test_fixed)}] en cours...')

    img_path = item['images'][0]
    gold_raw = json.loads(item['conversations'][1]['content'])
    gold     = {e: normalize(gold_raw.get(e, '')) for e in ENTITIES}

    pred_raw = inference(img_path)
    pred     = {e: normalize(pred_raw.get(e, '')) for e in ENTITIES}

    for entity in ENTITIES:
        g = gold[entity]
        p = pred[entity]

        if g == '' and p == '':
            pass  # vrai négatif → ignore
        elif g != '' and p == g:
            tp[entity] += 1
        elif g != '' and p != g:
            fn[entity] += 1
            if p != '':
                fp[entity] += 1
        elif g == '' and p != '':
            fp[entity] += 1

    results_detail.append({
        'idx'      : idx,
        'image'    : Path(img_path).name,
        'gold'     : gold_raw,
        'pred'     : pred_raw,
        'correct'  : {e: normalize(gold_raw.get(e,'')) == normalize(pred_raw.get(e,''))
                      for e in ENTITIES}
    })

# ── Calcul métriques ──────────────────────────────────────
print(f'\n{"="*60}')
print(f'📊 RÉSULTATS ÉVALUATION — GLM-OCR fine-tuné (v7s7-ep13)')
print(f'{"="*60}')
print(f'\n{"ENTITÉ":<20} {"PRECISION":>10} {"RECALL":>10} {"F1":>10} {"TP":>6} {"FP":>6} {"FN":>6}')
print('-'*60)

macro_p = macro_r = macro_f1 = 0
n_entities = 0

for entity in ENTITIES:
    t = tp[entity]
    f_p = fp[entity]
    f_n = fn[entity]

    precision = t / (t + f_p) if (t + f_p) > 0 else 0.0
    recall    = t / (t + f_n) if (t + f_n) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    macro_p  += precision
    macro_r  += recall
    macro_f1 += f1
    n_entities += 1

    bar = '█' * int(f1 * 20)
    print(f'{entity:<20} {precision:>9.1%} {recall:>10.1%} {f1:>9.1%}  {t:>5} {f_p:>6} {f_n:>6}  {bar}')

macro_p  /= n_entities
macro_r  /= n_entities
macro_f1 /= n_entities

print('-'*60)
print(f'{"MACRO AVG":<20} {macro_p:>9.1%} {macro_r:>10.1%} {macro_f1:>9.1%}')
print(f'\n{"="*60}')
print(f'🏆 Macro F1 Score : {macro_f1:.4f} ({macro_f1*100:.2f}%)')
print(f'   Macro Precision : {macro_p*100:.2f}%')
print(f'   Macro Recall    : {macro_r*100:.2f}%')
print(f'{"="*60}')

# ── Exact Match ───────────────────────────────────────────
exact_match = sum(
    1 for r in results_detail
    if all(r['correct'][e] for e in ENTITIES)
) / len(results_detail)

print(f'\n📋 Exact Match (toutes entités correctes) : {exact_match*100:.2f}%')
print(f'   Exemples évalués : {len(results_detail)}')
print(f'   Erreurs inférence : {errors}')

# ── Exemples erreurs ──────────────────────────────────────
print(f'\n📋 Exemples avec erreurs (5 premiers) :')
wrong = [r for r in results_detail if not all(r['correct'][e] for e in ENTITIES)]
for r in wrong[:5]:
    print(f'\n   Image : {r["image"]}')
    for e in ENTITIES:
        g = normalize(r['gold'].get(e, ''))
        p = normalize(r['pred'].get(e, ''))
        ok = '✅' if g == p else '❌'
        if g != p:
            print(f'   {ok} {e:<20} gold={g!r:<30} pred={p!r}')

# ── Sauvegarder résultats ─────────────────────────────────
results_summary = {
    'model'      : 'glmocr-v7s7-best-ep13',
    'test_size'  : len(test_fixed),
    'macro_f1'   : round(macro_f1, 4),
    'macro_p'    : round(macro_p, 4),
    'macro_r'    : round(macro_r, 4),
    'exact_match': round(exact_match, 4),
    'per_entity' : {}
}

for entity in ENTITIES:
    t   = tp[entity]
    f_p = fp[entity]
    f_n = fn[entity]
    p_  = t / (t + f_p) if (t + f_p) > 0 else 0.0
    r_  = t / (t + f_n) if (t + f_n) > 0 else 0.0
    f_  = 2 * p_ * r_ / (p_ + r_) if (p_ + r_) > 0 else 0.0
    results_summary['per_entity'][entity] = {
        'precision': round(p_, 4),
        'recall'   : round(r_, 4),
        'f1'       : round(f_, 4),
        'tp': t, 'fp': f_p, 'fn': f_n
    }

with open('/kaggle/working/evaluation_results.json', 'w') as f:
    json.dump(results_summary, f, ensure_ascii=False, indent=2)

print(f'\n✅ Résultats sauvegardés : /kaggle/working/evaluation_results.json')

GPU : Tesla T4
Images indexées : 4004

⏳ Chargement modèle depuis glmocr-v7s7-best-ep13...


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

✅ Modèle chargé
Test set : 801 exemples

🔍 Évaluation sur 801 exemples...

   [0/801] en cours...
   [50/801] en cours...
   [100/801] en cours...
   [150/801] en cours...
   [200/801] en cours...
   [250/801] en cours...
   [300/801] en cours...
   [350/801] en cours...
   [400/801] en cours...
   [450/801] en cours...
   [500/801] en cours...
   [550/801] en cours...
   [600/801] en cours...
   [650/801] en cours...
   [700/801] en cours...
   [750/801] en cours...
   [800/801] en cours...

📊 RÉSULTATS ÉVALUATION — GLM-OCR fine-tuné (v7s7-ep13)

ENTITÉ                PRECISION     RECALL         F1     TP     FP     FN
------------------------------------------------------------
CLIENT                   97.3%      97.3%     97.3%    779     22     22  ███████████████████
FOURNISSEUR              98.3%      98.3%     98.3%    787     14     14  ███████████████████
DATE_FACTURE             97.1%      97.1%     97.1%    778     23     23  ███████████████████
MONTANT_TTC              97.

## 3. Inference time (10 invoices)

In [ ]:
import time, torch
from PIL import Image
from pathlib import Path

# Tester sur 10 images seulement
test_sample = test_fixed[:10]
times = []

print('⏱️  Mesure temps inférence sur 10 images...\n')

for i, item in enumerate(test_sample):
    img_path = item['images'][0]
    
    start = time.time()
    pred  = inference(img_path)
    end   = time.time()
    
    elapsed = end - start
    times.append(elapsed)
    print(f'   image {i+1:02d} : {elapsed:.2f}s → {pred}')

print(f'\n{"="*45}')
print(f'⏱️  Temps moyen par image : {sum(times)/len(times):.2f}s')
print(f'   Temps min             : {min(times):.2f}s')
print(f'   Temps max             : {max(times):.2f}s')
print(f'\n📊 Estimation production :')
print(f'   1 facture   → ~{sum(times)/len(times):.1f}s')
print(f'   10 factures → ~{sum(times)/len(times)*10:.0f}s')
print(f'   100 factures→ ~{sum(times)/len(times)*100/60:.1f} min')
print(f'{"="*45}')

⏱️  Mesure temps inférence sur 10 images...

   image 01 : 5.10s
   image 02 : 5.74s
   image 03 : 4.27s
   image 04 : 5.05s
   image 05 : 5.11s
   image 06 : 4.76s
   image 07 : 4.71s
   image 08 : 4.59s
   image 09 : 4.51s
   image 10 : 4.96s

⏱️  Temps moyen par image : 4.88s
   Temps min             : 4.27s
   Temps max             : 5.74s

📊 Estimation production :
   1 facture   → ~4.9s
   10 factures → ~49s
   100 factures→ ~8.1 min
